In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import arviz as az

import sys
import os
import pickle
from tqdm import tqdm
# Get the absolute path to the folder containing `utils`
utils_path = os.path.abspath('../')
if utils_path not in sys.path:
    sys.path.append(utils_path)

os.environ["CUDA_VISIBLE_DEVICES"] = "1" # second gpu
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"]="platform"

from utils import *

# az.style.use("arviz-docgrid")
plt.rcParams['figure.dpi'] = 140

experiment_orientations = [159, 123, 87, 51, 15]
subjects = ["01", "02", "03", "04", "05", "06", "07" ,"09", "10", "11", "12"]
median_key = {15:0, 51:1, 87:2, 123:3, 159:4}
std_key = {15:0, 51:1, 87:2, 123:3, 159:4}

c_table = pd.read_csv('../data_caches/ctiraltable.csv')
med = np.load('../data_caches/med.npy')
std = np.load('../data_caches/std.npy')
ctimetable = np.load('../data_caches/ctimetable.npy')
r_table = pd.read_csv('../data_caches/rtrialtable.csv', index_col=0)
(x, y, d, r, e, cd, ce) = np.load('../data_caches/rtimetable.npy', allow_pickle=True)

In [2]:
import jax
jax.config.update('jax_platform_name', 'gpu')

import jax.numpy as jnp
import jax.random as jr
from jax import lax
from jax import vmap
import optax

from jax.extend import backend
print(backend.get_backend().platform)

gpu


In [3]:
def train_hmm(model, emissions, inputs, verbose = True):
    parameters, properties = model.initialize(key=jr.PRNGKey(1), method="prior", emissions = emissions)
    fit_params, lps = model.fit_em(params = parameters, props = properties, emissions = emissions, inputs = inputs, num_iters = 10,verbose = verbose)
    return model, fit_params, lps

def scale_to_bounds(x, axis=0, eps=1e-3):
    """
    Linearly rescales x along `axis` so that
      min → eps, max → 1-eps,
    and everything else is in between.
    """
    # compute min and max (keepdims so we can broadcast)
    mn = x.min(axis=axis, keepdims=True)
    mx = x.max(axis=axis, keepdims=True)
    # normalize to [0,1]
    x0 = (x - mn) / (mx - mn)
    # stretch to [eps, 1-eps]
    return x0 * (1 - 2*eps) + eps

## Data

In [4]:
# Modeling setup:
# emissions : 3 components -> reaction time, response angle, response error

# inputs for them:
# reaction time: error, surprise | attention, coherence, expectiation | bias
# response angle: stim, prev. stim, pre. resp | attention, coherence, expectation | bias
# response error: | attention, coherence, expectation | bias

# models for them:
# reaction time: Gamma GLM
# response angle: von Mises GLM
# response error: Beta GLM

In [5]:
rdf = r_table.copy()

# 1) Compute previous-trial target & response
rdf['prev_target'] = rdf.groupby(['subj', 'sess', 'run'])['t_calib'].shift(1)
rdf['prev_resp']   = rdf.groupby(['subj', 'sess', 'run'])['resp'].shift(1)
rdf['prev_target'] = rdf['prev_target'].fillna(rdf['t_calib'])
rdf['prev_resp']   = rdf['prev_resp'].fillna(rdf['resp'])

# 2) Reshape metadata into (n_trials, T, 1)
n_trials = rdf.shape[0] // 120
T = 120
reshape = lambda x: x.values.reshape(n_trials, T, 1).astype(np.float32)

baseline_bias    = np.ones((n_trials, T, 1), dtype=np.float32)
attention        = (rdf['att'] == "focused").astype(np.float32)
coherence        = (rdf['coh'] == "high").astype(np.float32)
expectation      = (rdf['exp_al'] == "expected").astype(np.float32)

attention   = reshape(attention)
coherence   = reshape(coherence)
expectation = reshape(expectation)

# 3) Error predictor for RT: signed error in radians
resp_err = np.deg2rad(rdf['resp_err'])
resp_err = reshape(resp_err)

# 4) Stimuli and history for RA
stim       = np.deg2rad(rdf['t_calib'])
prev_stim  = np.deg2rad(rdf['prev_target'])
prev_resp  = np.deg2rad(rdf['prev_resp'])

# wrap to [-pi, pi]
wrap = lambda x: ((x + np.pi) % (2*np.pi)) - np.pi
stim      = wrap(reshape(stim))
prev_stim = wrap(reshape(prev_stim))
prev_resp = wrap(reshape(prev_resp))

# 5) Reaction time (ms)
rt_raw = (rdf['f_resp'].values - 250).reshape(n_trials, T, 1) * (1000/120)
reaction_time = np.clip(rt_raw, 0.01, None).astype(np.float32)

# 6) Response angle in radians, wrapped
response_angle = np.deg2rad(rdf['resp']).values
response_angle = wrap(response_angle.reshape(n_trials, T, 1))

# 7) Response error for emissions (positive)
err_raw = rdf['resp_err'].values.reshape(n_trials, T, 1)
response_error = np.clip(err_raw, 0.01, None).astype(np.float32)

# 8) Build feature blocks
# RT features: [1, Error, Att, Coh, Exp, Error*Att, Error*Coh, Error*Exp]
x_rt = np.concatenate([
    baseline_bias,
    resp_err,
    attention, coherence, expectation,
    resp_err * attention,
    resp_err * coherence,
    resp_err * expectation
], axis=-1)

# RA features: [1, stim, prev_stim, prev_resp, Att, Coh, Exp]
x_ra = np.concatenate([
    baseline_bias,
    stim, prev_stim, prev_resp,
    attention, coherence, expectation
], axis=-1)

# RE features: [1, RT, Att, Coh, Exp, RT*Att, RT*Coh, RT*Exp]
x_re = np.concatenate([
    baseline_bias,
    reaction_time,
    attention, coherence, expectation,
    reaction_time * attention,
    reaction_time * coherence,
    reaction_time * expectation
], axis=-1)

# 9) Combine inputs and emissions
# inputs = jnp.concatenate([x_rt, x_ra, x_re], axis=-1)    # shape (n_trials, T, 23)
# emissions = jnp.concatenate([
#     reaction_time,
#     response_angle,
#     response_error
# ], axis=-1)                                           # shape (n_trials, T, 3)

inputs = jnp.array(x_rt)
emissions = jnp.array(reaction_time)

# Verify shapes
print("inputs  shape:", inputs.shape)
print("emissions shape:", emissions.shape)


inputs  shape: (288, 120, 8)
emissions shape: (288, 120, 1)


In [6]:

# print and check if any of these have nan in them
print("coherence", np.isnan(coherence).sum())
print("attention", np.isnan(attention).sum())
print("expectation", np.isnan(expectation).sum())

print("resp_err", np.isnan(resp_err).sum())

print("stim", np.isnan(stim).sum())
print("prev_stim", np.isnan(prev_stim).sum())
print("prev_resp", np.isnan(prev_resp).sum())

print("reaction_time", np.isnan(reaction_time).sum())
print("response_angle", np.isnan(response_angle).sum())
print("response_error", np.isnan(response_error).sum())

coherence 0
attention 0
expectation 0
resp_err 0
stim 0
prev_stim 0
prev_resp 0
reaction_time 0
response_angle 0
response_error 0


## Global

In [7]:
# num_states, input_dim, emission_dim = 2, 23, 3
num_states, input_dim, emission_dim = 2, 8, 1

In [8]:
# # Multi-state model cross-validation
# global_crossval = {}

# for nstate in tqdm(range(2, 3)):
#     model = BlockHMMrt(nstate, input_dim, emission_dim)
#     # make index of emissions size and shufflei t
#     shuffled_indices = np.random.permutation(emissions.shape[0])
#     parameters, properties = model.initialize(key=jr.PRNGKey(nstate), method="kmeans", emissions = emissions[shuffled_indices])
#     ll_mean, ll = cross_validate_regressor(model=model, emissions=emissions[shuffled_indices], key=jr.PRNGKey(0), num_iters=1, inputs=inputs[shuffled_indices])
#     global_crossval[nstate] = ll

In [9]:
# with open('./caches/global_crossval.pkl', 'wb') as f:
#     pickle.dump(global_crossval, f)
    
# with open('./caches/global_crossval.pkl', 'rb') as f:
#     global_crossval = pickle.load(f)

## Full training the global params

In [10]:
# global_params = {}

# losses = []

# for nstate in tqdm(range(2, 10)):
#     model = BlockHMMrt(nstate, input_dim, emission_dim)
#     parameters, properties = model.initialize(key=jr.PRNGKey(1), method="prior", emissions = emissions)
#     subset_idx = np.random.choice(np.arange(0, len(emissions)), len(emissions), replace=False)
#     fit_params, lps = model.fit_em(params = parameters, props = properties, emissions = emissions[subset_idx], inputs = inputs[subset_idx], num_iters = 500, verbose = False)
#     global_params[nstate] = fit_params
#     losses.append(lps)

In [11]:
# with open('./caches/global_params.pkl', 'wb') as f:
#     pickle.dump(global_params, f)
    
# with open('./caches/global_params.pkl', 'rb') as f:
#     global_params = pickle.load(f)

## Crossvalidate subject models LOO

In [12]:
# One state model cross-validation
# One state model does not need global since there is no state switching
one_crossval_results = {}

for idx, (s_em, s_in) in tqdm(enumerate(zip(emissions.reshape(12, 4 * 6, 120, 1), inputs.reshape(12, 4 * 6, 120, 8)))):
    one_crossval_results[idx] = {}
    shuffle_idx = np.random.permutation(len(s_em))
    model = BlockOneRT()
    ll_mean, ll = cross_validate_regressor(model=model, emissions=s_em[shuffle_idx], key=jr.PRNGKey(0), num_iters=5, inputs=s_in[shuffle_idx], init="default")
    one_crossval_results[idx][1] = ll

0it [00:00, ?it/s]

12it [00:09,  1.25it/s]


In [13]:
# with open('./caches/one_state_crossval_result.pkl', 'wb') as f:
#     pickle.dump(one_crossval_results, f)
    
# with open('./caches/one_state_crossval_result.pkl', 'rb') as f:
#     one_crossval_results = pickle.load(f)

In [14]:
# crossval_results = {}

# for idx, (s_em, s_in) in enumerate(zip(emissions.reshape(12, 4 * 6, 120, 3), inputs.reshape(12, 4 * 6, 120, 17))):
#     crossval_results[idx] = {}
#     shuffle_idx = np.random.permutation(len(s_em))
    
#     for nstate in tqdm(range(2, 10), desc=f'sub-{idx+1}'):
#         model = BlockHMM(nstate, input_dim, emission_dim)
#         gpar = global_params[nstate]
#         parameters, properties = model.initialize(key=jr.PRNGKey(1), 
#                                                   method="prior",
#                                                   initial_probs=gpar.initial.probs,
#                                                   transition_matrix=gpar.transitions.transition_matrix,
#                                                   weights_rt = gpar.emissions.weights_rt,
#                                                   alpha_rt = gpar.emissions.alpha_rt,
#                                                   weights_ra = gpar.emissions.weights_ra,
#                                                   kappa_ra = gpar.emissions.kappa_ra,
#                                                   weights_re = gpar.emissions.weights_re,
#                                                   phi_re = gpar.emissions.phi_re)
#         ll_mean, ll = cross_validate_regressor(model=model, emissions=s_em[shuffle_idx], key=jr.PRNGKey(0), num_iters=500, inputs=s_in[shuffle_idx], init = (parameters, properties))
#         crossval_results[idx][nstate] = ll

In [15]:
# with open('./caches/sub_crossval_results.pkl', 'wb') as f:
#     pickle.dump(crossval_results, f)
# with open('./caches/sub_crossval_results.pkl', 'rb') as f:
#     crossval_results = pickle.load(f)

## Subject params full training

In [16]:
# subject_params = {}

# for idx, (s_em, s_in) in enumerate(zip(emissions.reshape(12, 4 * 6, 120, 3), inputs.reshape(12, 4 * 6, 120, 17))):
#     subject_params[idx] = {}
#     shuffle_idx = np.random.permutation(len(s_em))
    
#     nstate = 2
#     model = BlockHMM(nstate, input_dim, emission_dim)
#     gpar = global_params[nstate]
#     parameters, properties = model.initialize(key=jr.PRNGKey(1), 
#                                                 method="prior",
#                                                 initial_probs=gpar.initial.probs,
#                                                 transition_matrix=gpar.transitions.transition_matrix,
#                                                 weights_rt = gpar.emissions.weights_rt,
#                                                 alpha_rt = gpar.emissions.alpha_rt,
#                                                 weights_ra = gpar.emissions.weights_ra,
#                                                 kappa_ra = gpar.emissions.kappa_ra,
#                                                 weights_re = gpar.emissions.weights_re,
#                                                 phi_re = gpar.emissions.phi_re)
#     fit_params, lps = model.fit_em(params = parameters, props = properties, emissions = s_em[shuffle_idx], inputs = s_in[shuffle_idx], num_iters = 500, verbose = False)
#     subject_params[idx][nstate] = fit_params

In [17]:
# with open('./caches/subject_params.pkl', 'wb') as f:
#     pickle.dump(subject_params, f)
    
# with open('./caches/subject_params.pkl', 'rb') as f:
#     subject_params = pickle.load(f)

## Most likely states

In [18]:
# ml_states = {}
# nstate = 2

# cem = emissions.reshape(12, 4 * 6, 120, 3)
# cin = inputs.reshape(12, 4 * 6, 120, 17)
# for idx in tqdm(range(12), desc=f''):
#     shuffle_idx = np.random.permutation(len(cem[idx]))
    
#     em = jnp.array(cem[idx].reshape(-1, 120, 3)[shuffle_idx])
#     inps = jnp.array(cin[idx].reshape(-1, 120, 17)[shuffle_idx])
    
#     model = BlockHMM(nstate, input_dim, emission_dim)
    
#     gpar = subject_params[idx][nstate]
#     parameters, properties = model.initialize(key=jr.PRNGKey(1), 
#                                                 method="prior",
#                                                 initial_probs=gpar.initial.probs,
#                                                 transition_matrix=gpar.transitions.transition_matrix,
#                                                 weights_rt = gpar.emissions.weights_rt,
#                                                 alpha_rt = gpar.emissions.alpha_rt,
#                                                 weights_ra = gpar.emissions.weights_ra,
#                                                 kappa_ra = gpar.emissions.kappa_ra,
#                                                 weights_re = gpar.emissions.weights_re,
#                                                 phi_re = gpar.emissions.phi_re)
#     # fit_params, lps = model.fit_em(params = parameters, props = properties, emissions = em[shuffle_idx], inputs = inps[shuffle_idx], num_iters = 100, verbose = False)
#     # Define a vmapped version of most_likely_states.
#     # This applies the function to each trial in the batch dimension of `em`.
#     most_likely_states_vmap = vmap(lambda trial, inp: model.most_likely_states(parameters, trial, inp))
#     t_trail = most_likely_states_vmap(em, inps)
#     ml_states[idx] = t_trail

In [19]:
# with open('./caches/ml_states.pkl', 'wb') as f:
#     pickle.dump(ml_states, f)
    
# with open('./caches/ml_states.pkl', 'rb') as f:
#     ml_states = pickle.load(f)